In [ ]:
"""
Speech Processing
=================
Demonstrates speech recognition (ASR) and text-to-speech (TTS) capabilities.

openai-whisper: https://github.com/openai/whisper
gTTS: https://gtts.readthedocs.io/
torchaudio: https://pytorch.org/audio/
SpeechRecognition: https://github.com/Uberi/speech_recognition

Uses synthetically generated audio — no external files required.
"""

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
import torch
import torchaudio
import soundfile as sf

In [ ]:
OUTPUT_DIR = os.path.join(os.path.dirname(__file__), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# =============================================================================
# Generate Synthetic Audio (sine wave with harmonics)
# =============================================================================
print("=" * 60)
print("Generating Synthetic Audio")
print("=" * 60)

In [ ]:
sample_rate = 16000
duration = 3.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

In [ ]:
# Create a multi-tone signal (A4 chord with harmonics)
frequencies = [440.0, 554.37, 659.25]  # A4, C#5, E5
signal = np.zeros_like(t)
for freq in frequencies:
    signal += 0.3 * np.sin(2 * np.pi * freq * t)

In [ ]:
# Add fade in/out
fade_len = int(0.1 * sample_rate)
fade_in = np.linspace(0, 1, fade_len)
fade_out = np.linspace(1, 0, fade_len)
signal[:fade_len] *= fade_in
signal[-fade_len:] *= fade_out

In [ ]:
# Normalize
signal = signal / np.max(np.abs(signal)) * 0.9

In [ ]:
audio_path = os.path.join(OUTPUT_DIR, 'speech_synthetic_audio.wav')
sf.write(audio_path, signal.astype(np.float32), sample_rate)

In [ ]:
print(f"Sample rate  : {sample_rate} Hz")
print(f"Duration     : {duration} s")
print(f"Frequencies  : {frequencies} Hz")
print(f"Signal shape : {signal.shape}")
print(f"Saved        : {audio_path}")

In [ ]:
# =============================================================================
# Whisper ASR — Transcribe with tiny model
# =============================================================================
print("\n" + "=" * 60)
print("Whisper ASR — Tiny Model")
print("=" * 60)

In [ ]:
try:
    import whisper

    print("Loading whisper tiny model (downloads ~75 MB on first use)...")
    model = whisper.load_model('tiny')

    # Transcribe the synthetic audio (will produce minimal results since it's tones)
    result = model.transcribe(audio_path, language='en')
    transcription = result['text'].strip()

    print(f"Model        : tiny")
    print(f"Language     : {result.get('language', 'en')}")
    print(f"Transcription: {transcription!r}")

    # Show segments if any
    if result.get('segments'):
        print(f"Segments     : {len(result['segments'])}")
        for seg in result['segments'][:3]:
            print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text'].strip()!r}")

    print("\nNote: Synthetic tones produce minimal transcription — this verifies the pipeline works.")

In [ ]:
except Exception as e:
    print(f"Whisper transcription skipped: {e}")

In [ ]:
# =============================================================================
# gTTS — Text to Speech
# =============================================================================
print("\n" + "=" * 60)
print("gTTS — Google Text-to-Speech")
print("=" * 60)

In [ ]:
# gTTS calls Google's TTS endpoint, so it only works with network access.
# Network problems are reported and skipped; any other error is a real bug
# and propagates.
from gtts import gTTS
from gtts.tts import gTTSError
import requests

In [ ]:
try:
    text = "Hello, this is a test of the speech synthesis system."
    tts = gTTS(text=text, lang='en', slow=False)

    gtts_path = os.path.join(OUTPUT_DIR, 'speech_gtts_output.mp3')
    tts.save(gtts_path)

    print(f"Input text   : {text!r}")
    print(f"Language     : en")
    print(f"Saved        : {gtts_path}")
    print(f"File size    : {os.path.getsize(gtts_path)} bytes")

In [ ]:
except (gTTSError, requests.RequestException) as e:
    print(f"gTTS synthesis skipped (network/service unavailable): {e}")

In [ ]:
# =============================================================================
# torchaudio — Waveform Analysis and Transforms
# =============================================================================
print("\n" + "=" * 60)
print("torchaudio — Waveform Analysis")
print("=" * 60)

In [ ]:
waveform = torch.tensor(signal, dtype=torch.float32).unsqueeze(0)
print(f"Waveform tensor : {waveform.shape}, dtype={waveform.dtype}")

In [ ]:
# Spectrogram
spectrogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=512, hop_length=128, power=2.0
)
spec = spectrogram_transform(waveform)
print(f"Spectrogram     : {spec.shape} (channels x freq_bins x frames)")

In [ ]:
# Mel spectrogram
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate, n_fft=512, hop_length=128, n_mels=64
)
mel_spec = mel_transform(waveform)
print(f"Mel Spectrogram : {mel_spec.shape} (channels x mel_bins x frames)")

In [ ]:
# AmplitudeToDB
to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
spec_db = to_db(spec)
mel_db = to_db(mel_spec)

In [ ]:
# =============================================================================
# SpeechRecognition — Library Verification
# =============================================================================
print("\n" + "=" * 60)
print("SpeechRecognition — Library Info")
print("=" * 60)

In [ ]:
try:
    import speech_recognition as sr

    recognizer = sr.Recognizer()
    print(f"Library version : {sr.__version__}")
    print(f"Recognizer      : {type(recognizer).__name__}")
    print(f"Energy threshold: {recognizer.energy_threshold}")

    # Load the synthetic audio file
    with sr.AudioFile(audio_path) as source:
        audio_data = recognizer.record(source)
    print(f"Audio duration  : {len(audio_data.frame_data) / audio_data.sample_rate / audio_data.sample_width:.2f} s")

In [ ]:
except Exception as e:
    print(f"SpeechRecognition verification skipped: {e}")

In [ ]:
# =============================================================================
# Visualization — Waveform and Spectrogram
# =============================================================================
print("\n" + "=" * 60)
print("Visualization — Waveform and Spectrogram")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

In [ ]:
# Waveform
times = np.linspace(0, duration, len(signal))
axes[0].plot(times, signal, linewidth=0.5, color='steelblue')
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Synthetic Audio Waveform (A Major Chord)")

In [ ]:
# Spectrogram
axes[1].imshow(
    spec_db[0].numpy(),
    aspect='auto',
    origin='lower',
    cmap='magma',
    extent=[0, duration, 0, sample_rate / 2],
)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title("Spectrogram (dB)")

In [ ]:
# Mel spectrogram
axes[2].imshow(
    mel_db[0].numpy(),
    aspect='auto',
    origin='lower',
    cmap='magma',
    extent=[0, duration, 0, 64],
)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Mel Band")
axes[2].set_title("Mel Spectrogram (dB)")

In [ ]:
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'speech_waveforms.png')
plt.savefig(plot_path, dpi=150)
print(f"Saved: {plot_path}")
plt.close()

In [ ]:
# =============================================================================
# Summary
# =============================================================================
print("\n" + "=" * 60)
print("Speech processing complete!")
print("Outputs: speech_synthetic_audio.wav, speech_gtts_output.mp3,")
print("         speech_waveforms.png")
print("=" * 60)